# BinSight — waste image classifier

Transfer-learning pipeline that trains a six-class waste classifier on
[TrashNet](https://github.com/garythung/trashnet) and exports a validated
LiteRT/TFLite model for the BinSight iOS app.

---

## Honesty notice

**No cell in this notebook has been executed by its author.** It was written and
statically validated (valid notebook JSON, every code cell parses under
`ast.parse`), but it has never been run, so the repository contains **no metrics,
no plots and no `.tflite` file** yet.

Anything you see below is *code*, not *results*. Once you run this notebook the
outputs it produces are real measurements — until then, treat every number in
`docs/MODEL_CARD.md` as unfilled.

## Where to run it

Google Colab, T4 GPU runtime. The pinned environment matches the Colab default
runtime (Python 3.12, TensorFlow 2.19). See `ml/README.md` for the exact steps
and for how to copy the artefacts back into the repository.

## What it does

1. Downloads TrashNet (42.8 MB resized archive) from the official repository.
2. Validates every image and reports unreadable files.
3. Builds a deterministic stratified 70/15/15 split, asserts no file overlap,
   and saves a split manifest.
4. Fine-tunes EfficientNet-Lite0 (ImageNet) in two stages.
5. Evaluates **once** on the held-out test split.
6. Exports float32 / float16 / int8 TFLite, validates each with the LiteRT
   interpreter, compares them numerically against Keras, and selects one.

## 1. Environment

Pinned deliberately. The notes on *why* each pin was chosen are in
`ml/README.md`; the short version:

* **TensorFlow 2.19 / Keras 3** — the Colab default. Installing a different
  TensorFlow into Colab costs several minutes and frequently breaks CUDA, so the
  notebook adopts what is already there instead of fighting it.
* **keras-hub 0.30.0** — supplies `efficientnet_lite0_ra_imagenet`, the only
  officially packaged EfficientNet-Lite0 with ImageNet weights in the Keras 3
  toolchain. `tflite-model-maker` is abandoned and is deliberately **not** used.
* **tflite-support** is attempted but expected to fail: its newest release
  (0.4.4) ships no cp312 wheel, and the Colab runtime is Python 3.12. The
  notebook falls back to `labels.txt` + `model_contract.json` and says so.

In [ ]:
# Colab already provides tensorflow, numpy, scikit-learn, pandas and matplotlib.
# Only keras-hub is missing.
%pip install -q "keras-hub==0.30.0"
print("keras-hub installed. If Colab asks you to restart the session, do it and re-run from here.")

In [ ]:
import os
import ast  # noqa: F401  (kept so a static parse of this notebook exercises stdlib imports)

# Keras 3 with the TensorFlow backend: needed because we export a TF SavedModel
# on the way to TFLite.
os.environ["KERAS_BACKEND"] = "tensorflow"

import json
import hashlib
import pathlib
import platform
import random
import shutil
import time
import urllib.request
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import keras_hub
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "keras_hub": keras_hub.__version__,
    "numpy": np.__version__,
    "gpus": [d.name for d in tf.config.list_physical_devices("GPU")],
}
for key, value in ENVIRONMENT.items():
    print(f"{key:12s}: {value}")

if not ENVIRONMENT["gpus"]:
    print("\nWARNING: no GPU visible. Training will work but will be slow.")
    print("In Colab: Runtime > Change runtime type > T4 GPU.")

## 2. Configuration

Every hyperparameter lives in this one cell.

In [ ]:
CONFIG = {
    # Reproducibility
    "seed": 42,
    # Full op determinism makes runs bit-identical but costs speed on GPU.
    # Off by default; the seed alone already pins the split and the shuffles.
    "enable_op_determinism": False,

    # Data
    "image_size": 224,          # EfficientNet-Lite0's native training resolution
    "batch_size": 32,
    "val_fraction": 0.15,
    "test_fraction": 0.15,

    # Backbone
    "backbone_preset": "efficientnet_lite0_ra_imagenet",

    # Head
    "head_dropout": 0.2,

    # Stage A - frozen backbone, train the head
    "stage_a_epochs": 25,
    "stage_a_lr": 1e-3,

    # Stage B - unfreeze the tail of the backbone and fine-tune.
    # Conditional: kept only if it actually beats Stage A on validation.
    "stage_b_enabled": True,
    "stage_b_epochs": 15,
    "stage_b_lr": 1e-4,
    "stage_b_unfreeze_layers": 40,

    "early_stopping_patience": 5,

    # Class weighting is applied only if the observed imbalance exceeds this
    # ratio (max class count / min class count). TrashNet sits around 4.3.
    "class_weight_trigger": 3.0,

    # Export
    "export_int8": True,
}

SEED = CONFIG["seed"]
IMG_SIZE = CONFIG["image_size"]

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)
if CONFIG["enable_op_determinism"]:
    tf.config.experimental.enable_op_determinism()

# Canonical class order. This is the contract the iOS app relies on: the index
# of a class here IS its output index in the exported model. It is asserted
# against the training pipeline further down, never assumed.
#
# Note it is deliberately NOT alphabetical - alphabetical order would put
# general_waste second.
CLASS_ORDER = ["cardboard", "glass", "metal", "paper", "plastic", "general_waste"]
NUM_CLASSES = len(CLASS_ORDER)

# TrashNet's on-disk folder names -> our internal machine-safe labels.
TRASHNET_TO_INTERNAL = {
    "cardboard": "cardboard",
    "glass": "glass",
    "metal": "metal",
    "paper": "paper",
    "plastic": "plastic",
    "trash": "general_waste",   # the only rename
}

# Internal label -> the string the app shows a user.
INTERNAL_TO_DISPLAY = {
    "cardboard": "Cardboard",
    "glass": "Glass",
    "metal": "Metal",
    "paper": "Paper",
    "plastic": "Plastic",
    "general_waste": "General waste",
}

assert sorted(TRASHNET_TO_INTERNAL.values()) == sorted(CLASS_ORDER)
assert set(INTERNAL_TO_DISPLAY) == set(CLASS_ORDER)

ARTIFACTS = pathlib.Path("artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
WORK = pathlib.Path("work")          # scratch; never committed
WORK.mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2))
print("\nCLASS_ORDER (output index order):")
for i, name in enumerate(CLASS_ORDER):
    print(f"  {i}: {name:14s} -> {INTERNAL_TO_DISPLAY[name]!r}")

## 3. Dataset — TrashNet

**Source:** <https://github.com/garythung/trashnet> —
`data/dataset-resized.zip` (42.8 MB, 512x384 JPEGs).

**Licence:** the TrashNet repository is MIT-licensed. Its README asks that
anyone using the dataset cite the repository, which `THIRD_PARTY_NOTICES.md`
does.

**Why the resized archive and not the 3.5 GB original:** the resized archive is
the one committed to the official repository, downloads in seconds, and is
already larger than our 224x224 input. Nothing is gained by pulling 3.5 GB of
higher-resolution source images through a Colab session.

The archive contains a `__MACOSX/` sidecar directory and `.DS_Store` files. Both
are filtered out below — counting them as images is an easy way to end up with a
class distribution that quietly disagrees with the paper.

In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/garythung/trashnet/master/data/dataset-resized.zip"
ARCHIVE = WORK / "dataset-resized.zip"
DATA_ROOT = WORK / "dataset-resized"

if not ARCHIVE.exists():
    print(f"Downloading {DATASET_URL} ...")
    started = time.time()
    urllib.request.urlretrieve(DATASET_URL, ARCHIVE)
    print(f"  done in {time.time() - started:.1f}s, {ARCHIVE.stat().st_size / 1e6:.1f} MB")
else:
    print(f"Archive already present ({ARCHIVE.stat().st_size / 1e6:.1f} MB)")

if not DATA_ROOT.exists():
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall(WORK)
    print(f"Extracted to {DATA_ROOT}")

# The archive ships a macOS sidecar directory; drop it so it can never be walked.
macosx = WORK / "__MACOSX"
if macosx.exists():
    shutil.rmtree(macosx)
    print("Removed __MACOSX/ sidecar directory")

print("\nClass folders found on disk:")
for child in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    print(f"  {child.name}")

### 3.1 Validate every image

Every file is opened and fully decoded. `Image.verify()` alone is not enough —
it checks the header but not the entropy-coded data, so a truncated JPEG passes
`verify()` and then explodes during training.

In [ ]:
records = []
corrupt = []

image_paths = sorted(
    p for p in DATA_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    and not p.name.startswith(".")          # .DS_Store and friends
)
print(f"Candidate image files: {len(image_paths)}")

for path in image_paths:
    folder = path.parent.name
    if folder not in TRASHNET_TO_INTERNAL:
        corrupt.append((str(path), f"unexpected class folder {folder!r}"))
        continue
    try:
        with Image.open(path) as im:
            im = im.convert("RGB")
            im.load()               # forces a full decode, not just the header
            width, height = im.size
        if width < 32 or height < 32:
            raise ValueError(f"implausibly small image {width}x{height}")
    except Exception as exc:        # noqa: BLE001 - we want to record anything
        corrupt.append((str(path), f"{type(exc).__name__}: {exc}"))
        continue

    label = TRASHNET_TO_INTERNAL[folder]
    records.append(
        {
            "relative_path": str(path.relative_to(DATA_ROOT)),
            "source_folder": folder,
            "label": label,
            "label_index": CLASS_ORDER.index(label),
            "width": width,
            "height": height,
        }
    )

frame = pd.DataFrame(records)

print(f"\nReadable images : {len(frame)}")
print(f"Unreadable files: {len(corrupt)}")
if corrupt:
    print("\nUnreadable / rejected files:")
    for path, reason in corrupt[:50]:
        print(f"  {path}: {reason}")
    if len(corrupt) > 50:
        print(f"  ... and {len(corrupt) - 50} more")
else:
    print("No corrupt or unreadable files found.")

assert len(frame) > 0, "No readable images - check the download and extraction steps."

In [ ]:
distribution = (
    frame.groupby(["label_index", "label"])
    .size()
    .reset_index(name="count")
    .sort_values("label_index")
)
distribution["share"] = distribution["count"] / len(frame)

print("Class distribution (discovered by code, not assumed):\n")
print(distribution.to_string(index=False, formatters={"share": "{:.1%}".format}))
print(f"\nTotal: {len(frame)} images across {frame['label'].nunique()} classes")

counts = distribution.set_index("label")["count"]
IMBALANCE_RATIO = counts.max() / counts.min()
print(f"\nImbalance ratio (max/min): {IMBALANCE_RATIO:.2f}")
print(f"  largest : {counts.idxmax()} ({counts.max()})")
print(f"  smallest: {counts.idxmin()} ({counts.min()})")

### 3.2 Deterministic stratified split

70 / 15 / 15, `random_state=42`, stratified on the label so the rare
`general_waste` class keeps its proportion in all three splits.

The overlap assertion is not decoration: a leak between train and test is the
single easiest way to publish an accuracy number that means nothing.

In [ ]:
train_frame, holdout = train_test_split(
    frame,
    test_size=CONFIG["val_fraction"] + CONFIG["test_fraction"],
    stratify=frame["label"],
    random_state=SEED,
    shuffle=True,
)

# Split the holdout in half -> equal validation and test fractions.
relative_test = CONFIG["test_fraction"] / (CONFIG["val_fraction"] + CONFIG["test_fraction"])
val_frame, test_frame = train_test_split(
    holdout,
    test_size=relative_test,
    stratify=holdout["label"],
    random_state=SEED,
    shuffle=True,
)

train_frame = train_frame.assign(split="train")
val_frame = val_frame.assign(split="val")
test_frame = test_frame.assign(split="test")

manifest = pd.concat([train_frame, val_frame, test_frame], ignore_index=True)
manifest = manifest.sort_values(["split", "label", "relative_path"]).reset_index(drop=True)

# --- No file may appear in more than one split -------------------------------
train_files = set(train_frame["relative_path"])
val_files = set(val_frame["relative_path"])
test_files = set(test_frame["relative_path"])

assert not (train_files & val_files), f"train/val overlap: {sorted(train_files & val_files)[:5]}"
assert not (train_files & test_files), f"train/test overlap: {sorted(train_files & test_files)[:5]}"
assert not (val_files & test_files), f"val/test overlap: {sorted(val_files & test_files)[:5]}"
assert len(train_files) + len(val_files) + len(test_files) == len(frame), "split sizes do not sum to the dataset"
assert len(manifest) == len(frame), "manifest lost or duplicated rows"
print("Split overlap assertions passed: the three splits are disjoint and complete.\n")

summary = (
    manifest.pivot_table(index="label", columns="split", values="relative_path", aggfunc="count")
    .reindex(CLASS_ORDER)[["train", "val", "test"]]
)
summary["total"] = summary.sum(axis=1)
print(summary.to_string())
print(f"\ntrain={len(train_frame)}  val={len(val_frame)}  test={len(test_frame)}")

MANIFEST_PATH = ARTIFACTS / "split_manifest.csv"
manifest[["relative_path", "source_folder", "label", "label_index", "split"]].to_csv(
    MANIFEST_PATH, index=False
)
print(f"\nSplit manifest written to {MANIFEST_PATH} (paths + labels + split, no image bytes)")

## 4. Input pipeline

Images stay in **[0, 255] float32 RGB** all the way through. Normalisation is a
layer *inside* the model (section 5), which means the exported TFLite file takes
raw pixel values and the iOS side has no preprocessing constants to get wrong.

Augmentation is tuned for handheld phone photos: small rotation and shift, mild
zoom, mild brightness/contrast, horizontal flip. No vertical flip and no
aggressive colour jitter — hue and saturation *are* the material cue that
separates glass from plastic, so distorting them would train the model to ignore
the very signal it needs.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_image(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_jpeg(raw, channels=3)          # RGB, uint8
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
    image = tf.cast(image, tf.float32)                   # still 0..255
    return image, label


def make_dataset(split_frame, *, shuffle, augment):
    paths = [str(DATA_ROOT / p) for p in split_frame["relative_path"]]
    labels = split_frame["label_index"].to_numpy(dtype=np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(CONFIG["batch_size"])
    if augment:
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


augmenter = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal", seed=SEED),
        keras.layers.RandomRotation(0.06, fill_mode="reflect", seed=SEED),
        keras.layers.RandomTranslation(0.08, 0.08, fill_mode="reflect", seed=SEED),
        keras.layers.RandomZoom(0.12, fill_mode="reflect", seed=SEED),
        keras.layers.RandomBrightness(0.15, value_range=(0.0, 255.0), seed=SEED),
        keras.layers.RandomContrast(0.15, seed=SEED),
    ],
    name="augmenter",
)

train_ds = make_dataset(train_frame, shuffle=True, augment=True)
val_ds = make_dataset(val_frame, shuffle=False, augment=False)
test_ds = make_dataset(test_frame, shuffle=False, augment=False)

sample_images, sample_labels = next(iter(train_ds))
print(f"batch images: {sample_images.shape} {sample_images.dtype}")
print(f"batch labels: {sample_labels.shape} {sample_labels.dtype}")
print(f"pixel range : [{float(tf.reduce_min(sample_images)):.1f}, {float(tf.reduce_max(sample_images)):.1f}]")

In [ ]:
# Visual sanity check on the augmentation strength: if these look destroyed,
# the augmentation is too aggressive for a material-classification task.
fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for ax, image, label in zip(axes.flat, sample_images, sample_labels):
    ax.imshow(np.clip(image.numpy() / 255.0, 0, 1))
    ax.set_title(CLASS_ORDER[int(label)], fontsize=9)
    ax.axis("off")
fig.suptitle("Augmented training batch", fontsize=12)
fig.tight_layout()
plt.show()

## 5. Model — EfficientNet-Lite0

`efficientnet_lite0_ra_imagenet` from KerasHub. EfficientNet-Lite drops the
squeeze-and-excite blocks and swaps swish for ReLU6 specifically so it converts
cleanly to mobile runtimes — which is exactly what we need downstream.

The normalisation constants are **read off the preset's own image converter**
rather than hardcoded, then printed and later written into
`model_contract.json`. Guessing them is a classic silent-accuracy-loss bug: the
model trains fine and then underperforms on device because the app normalised
differently from training.

In [ ]:
PRESET = CONFIG["backbone_preset"]

backbone = keras_hub.models.EfficientNetBackbone.from_preset(PRESET)
print(f"Loaded backbone {PRESET!r}: {backbone.count_params():,} parameters")

# --- Derive the preprocessing contract from the preset -----------------------
NORMALIZATION = {"source": None, "scale": None, "offset": None}
try:
    converter = keras_hub.layers.ImageConverter.from_preset(PRESET)
    scale = getattr(converter, "scale", None)
    offset = getattr(converter, "offset", None)
    if scale is None:
        raise AttributeError("image converter exposes no scale")
    NORMALIZATION = {
        "source": f"keras_hub.layers.ImageConverter.from_preset({PRESET!r})",
        "scale": np.atleast_1d(np.array(scale, dtype="float64")).tolist(),
        "offset": np.atleast_1d(np.array(offset if offset is not None else 0.0, dtype="float64")).tolist(),
    }
except Exception as exc:  # noqa: BLE001
    # Documented fallback: the timm "ra" recipe this preset was ported from uses
    # standard ImageNet mean/std on 0..1 inputs.
    print(f"Could not read the preset's converter ({type(exc).__name__}: {exc}).")
    print("Falling back to the documented ImageNet mean/std for the timm 'ra' recipe.")
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    NORMALIZATION = {
        "source": "fallback: ImageNet mean/std (timm 'ra' recipe)",
        "scale": (1.0 / (255.0 * std)).tolist(),
        "offset": (-mean / std).tolist(),
    }

print("\nResolved normalisation applied INSIDE the model (input is 0..255 RGB):")
print(json.dumps(NORMALIZATION, indent=2))

In [ ]:
def build_model():
    """0..255 RGB in, softmax probabilities out.

    Normalisation lives inside the graph so the exported TFLite model has a
    single, dead-simple contract for the iOS side.
    """
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype="float32", name="image")

    x = keras.layers.Rescaling(
        scale=NORMALIZATION["scale"],
        offset=NORMALIZATION["offset"],
        name="normalize",
    )(inputs)

    features = backbone(x)
    # KerasHub backbones return either a tensor or a dict of pyramid levels.
    if isinstance(features, dict):
        deepest = sorted(features.keys())[-1]
        print(f"Backbone returned a feature dict; using level {deepest!r}")
        features = features[deepest]

    x = keras.layers.GlobalAveragePooling2D(name="pool")(features)
    x = keras.layers.Dropout(CONFIG["head_dropout"], name="head_dropout")(x)
    outputs = keras.layers.Dense(
        NUM_CLASSES, activation="softmax", dtype="float32", name="probabilities"
    )(x)

    return keras.Model(inputs, outputs, name="binsight_waste_classifier")


backbone.trainable = False          # Stage A
model = build_model()
model.summary()

assert model.output_shape[-1] == NUM_CLASSES, (
    f"model outputs {model.output_shape[-1]} classes, expected {NUM_CLASSES}"
)
print(f"\nTrainable parameters (Stage A): {sum(np.prod(w.shape) for w in model.trainable_weights):,}")

In [ ]:
# --- Class weighting, applied only if the data actually justifies it ---------
if IMBALANCE_RATIO > CONFIG["class_weight_trigger"]:
    frequencies = counts.reindex(CLASS_ORDER).to_numpy(dtype="float64")
    weights = frequencies.sum() / (NUM_CLASSES * frequencies)
    CLASS_WEIGHT = {i: float(w) for i, w in enumerate(weights)}
    print(
        f"Imbalance ratio {IMBALANCE_RATIO:.2f} exceeds the trigger "
        f"({CONFIG['class_weight_trigger']}), so balanced class weights are applied:"
    )
    for i, name in enumerate(CLASS_ORDER):
        print(f"  {i}: {name:14s} n={int(frequencies[i]):4d}  weight={CLASS_WEIGHT[i]:.3f}")
else:
    CLASS_WEIGHT = None
    print(
        f"Imbalance ratio {IMBALANCE_RATIO:.2f} is below the trigger "
        f"({CONFIG['class_weight_trigger']}); training without class weights."
    )

## 6. Stage A — frozen backbone

Train only the head. Early stopping on validation accuracy with
`restore_best_weights=True`, so the weights we carry forward are the best epoch's
rather than the last epoch's.

In [ ]:
STAGE_A_CKPT = WORK / "stage_a.keras"

model.compile(
    optimizer=keras.optimizers.Adam(CONFIG["stage_a_lr"]),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

stage_a_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=CONFIG["early_stopping_patience"],
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        STAGE_A_CKPT, monitor="val_accuracy", save_best_only=True, verbose=0
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

stage_a_started = time.time()
history_a = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["stage_a_epochs"],
    class_weight=CLASS_WEIGHT,
    callbacks=stage_a_callbacks,
    verbose=2,
)
stage_a_seconds = time.time() - stage_a_started

stage_a_val_accuracy = float(max(history_a.history["val_accuracy"]))
print(f"\nStage A finished in {stage_a_seconds / 60:.1f} min")
print(f"Stage A best validation accuracy: {stage_a_val_accuracy:.4f}")

## 7. Stage B — fine-tune the backbone tail (conditional)

Unfreeze the last `stage_b_unfreeze_layers` layers at a 10x lower learning rate.

Two things worth calling out:

* **BatchNorm stays frozen.** With ~1 700 training images, letting BN recompute
  its statistics on small batches usually makes validation accuracy *worse*.
* **Stage B is kept only if it wins.** The comparison is made on validation
  accuracy; if fine-tuning does not help, the Stage A weights are restored. This
  is a measurement, not an assumption.

In [ ]:
stage_b_val_accuracy = None
stage_b_seconds = 0.0
history_b = None
used_stage_b = False

if CONFIG["stage_b_enabled"]:
    model.save_weights(WORK / "stage_a_weights.weights.h5")

    backbone.trainable = True
    frozen_bn = 0
    tail = backbone.layers[-CONFIG["stage_b_unfreeze_layers"]:]
    for layer in backbone.layers:
        layer.trainable = layer in tail
    for layer in backbone.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
            frozen_bn += 1

    trainable_count = sum(np.prod(w.shape) for w in model.trainable_weights)
    print(f"Unfroze the last {len(tail)} backbone layers ({frozen_bn} BatchNorm layers kept frozen)")
    print(f"Trainable parameters (Stage B): {trainable_count:,}")

    model.compile(
        optimizer=keras.optimizers.Adam(CONFIG["stage_b_lr"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    stage_b_started = time.time()
    history_b = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CONFIG["stage_b_epochs"],
        class_weight=CLASS_WEIGHT,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy",
                patience=CONFIG["early_stopping_patience"],
                restore_best_weights=True,
                verbose=1,
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1
            ),
        ],
        verbose=2,
    )
    stage_b_seconds = time.time() - stage_b_started
    stage_b_val_accuracy = float(max(history_b.history["val_accuracy"]))

    print(f"\nStage B finished in {stage_b_seconds / 60:.1f} min")
    print(f"Stage A best val accuracy: {stage_a_val_accuracy:.4f}")
    print(f"Stage B best val accuracy: {stage_b_val_accuracy:.4f}")

    if stage_b_val_accuracy > stage_a_val_accuracy:
        used_stage_b = True
        print("\n-> Stage B improved validation accuracy. Keeping the fine-tuned weights.")
    else:
        model.load_weights(WORK / "stage_a_weights.weights.h5")
        print("\n-> Stage B did NOT improve validation accuracy. Reverting to Stage A weights.")
else:
    print("Stage B disabled in CONFIG.")

SELECTED_STAGE = "stage_b_finetuned" if used_stage_b else "stage_a_frozen_backbone"
SELECTED_VAL_ACCURACY = stage_b_val_accuracy if used_stage_b else stage_a_val_accuracy
print(f"\nSelected model: {SELECTED_STAGE} (val accuracy {SELECTED_VAL_ACCURACY:.4f})")

In [ ]:
# --- Training curves ---------------------------------------------------------
def concatenated(metric):
    values = list(history_a.history.get(metric, []))
    if history_b is not None:
        values += list(history_b.history.get(metric, []))
    return values


fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
boundary = len(history_a.history.get("loss", []))

for ax, (metric, val_metric, title) in zip(
    axes,
    [("accuracy", "val_accuracy", "Accuracy"), ("loss", "val_loss", "Loss")],
):
    ax.plot(concatenated(metric), label="train", linewidth=1.8)
    ax.plot(concatenated(val_metric), label="validation", linewidth=1.8)
    if history_b is not None:
        ax.axvline(boundary - 0.5, color="grey", linestyle="--", linewidth=1)
        ax.text(boundary - 0.4, ax.get_ylim()[0], " Stage B", fontsize=8, color="grey")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(alpha=0.25)

fig.suptitle(
    f"BinSight training curves - selected: {SELECTED_STAGE}", fontsize=12
)
fig.tight_layout()
CURVES_PATH = ARTIFACTS / "training_curves.png"
fig.savefig(CURVES_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {CURVES_PATH}")

## 8. Evaluation — held-out test split, evaluated once

Model selection is finished. Everything above used only train and validation.
The test split is touched exactly once, here, and the guard below enforces
that.

In [ ]:
TEST_ALREADY_EVALUATED = globals().get("TEST_ALREADY_EVALUATED", False)
assert not TEST_ALREADY_EVALUATED, (
    "The test split has already been evaluated in this session. Re-running this "
    "cell after seeing the result would turn the test set into a validation set. "
    "Restart the runtime and re-run from the top if you need a fresh evaluation."
)

test_probabilities = model.predict(test_ds, verbose=0)
test_predictions = test_probabilities.argmax(axis=1)
test_truth = np.concatenate([y.numpy() for _, y in test_ds])

TEST_ALREADY_EVALUATED = True

assert len(test_truth) == len(test_frame), "test label count does not match the manifest"

test_accuracy = float(accuracy_score(test_truth, test_predictions))
macro_f1 = float(f1_score(test_truth, test_predictions, average="macro"))
weighted_f1 = float(f1_score(test_truth, test_predictions, average="weighted"))

print(f"Test images  : {len(test_truth)}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Macro F1     : {macro_f1:.4f}")
print(f"Weighted F1  : {weighted_f1:.4f}")
print("\nClassification report:\n")
print(classification_report(test_truth, test_predictions, target_names=CLASS_ORDER, digits=4))

In [ ]:
report_dict = classification_report(
    test_truth, test_predictions, target_names=CLASS_ORDER, output_dict=True, digits=6
)
report_frame = pd.DataFrame(report_dict).transpose()
REPORT_PATH = ARTIFACTS / "classification_report.csv"
report_frame.to_csv(REPORT_PATH)
print(f"Saved {REPORT_PATH}\n")
print(report_frame.to_string())

In [ ]:
def plot_confusion(matrix, title, path, *, normalized):
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    image = ax.imshow(matrix, cmap="magma" if normalized else "viridis")
    ax.set_xticks(range(NUM_CLASSES), CLASS_ORDER, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES), CLASS_ORDER)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)

    threshold = matrix.max() / 2.0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            value = matrix[i, j]
            ax.text(
                j, i,
                f"{value:.2f}" if normalized else f"{int(value)}",
                ha="center", va="center", fontsize=9,
                color="white" if value < threshold else "black",
            )
    fig.colorbar(image, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {path}")


confusion_counts = confusion_matrix(test_truth, test_predictions, labels=range(NUM_CLASSES))
plot_confusion(
    confusion_counts,
    "BinSight confusion matrix (test split, counts)",
    ARTIFACTS / "confusion_matrix.png",
    normalized=False,
)

with np.errstate(divide="ignore", invalid="ignore"):
    confusion_normalized = confusion_counts / confusion_counts.sum(axis=1, keepdims=True)
confusion_normalized = np.nan_to_num(confusion_normalized)
plot_confusion(
    confusion_normalized,
    "BinSight confusion matrix (test split, row-normalised)",
    ARTIFACTS / "confusion_matrix_normalized.png",
    normalized=True,
)

In [ ]:
# --- Correct and misclassified examples with their probabilities -------------
test_paths = [str(DATA_ROOT / p) for p in test_frame["relative_path"]]
confidences = test_probabilities.max(axis=1)
correct_mask = test_predictions == test_truth

rng = np.random.default_rng(SEED)
correct_idx = rng.choice(np.flatnonzero(correct_mask), size=min(6, int(correct_mask.sum())), replace=False)
wrong_pool = np.flatnonzero(~correct_mask)
wrong_idx = rng.choice(wrong_pool, size=min(6, len(wrong_pool)), replace=False) if len(wrong_pool) else np.array([], dtype=int)


def show_examples(indices, title, colour):
    if len(indices) == 0:
        print(f"{title}: none to show.")
        return
    columns = len(indices)
    fig, axes = plt.subplots(1, columns, figsize=(2.6 * columns, 3.6))
    axes = np.atleast_1d(axes)
    for ax, index in zip(axes, indices):
        with Image.open(test_paths[index]) as im:
            ax.imshow(im.convert("RGB"))
        ax.axis("off")
        ax.set_title(
            f"true: {CLASS_ORDER[test_truth[index]]}\n"
            f"pred: {CLASS_ORDER[test_predictions[index]]}\n"
            f"p={confidences[index]:.2f}",
            fontsize=8.5, color=colour,
        )
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


show_examples(correct_idx, "Correctly classified test examples", "darkgreen")
show_examples(wrong_idx, "Misclassified test examples", "darkred")

if len(wrong_pool):
    print("\nMost confident mistakes (these are the ones worth studying):")
    for index in wrong_pool[np.argsort(-confidences[wrong_pool])][:10]:
        print(
            f"  {test_frame.iloc[index]['relative_path']:34s} "
            f"true={CLASS_ORDER[test_truth[index]]:14s} "
            f"pred={CLASS_ORDER[test_predictions[index]]:14s} p={confidences[index]:.3f}"
        )

### 8.1 Expected domain shift — read this before trusting the number above

The test split above is drawn from the *same* distribution as the training data,
so it measures how well the model learned TrashNet. It does **not** measure how
well BinSight will work in a kitchen.

TrashNet images are studio-like: a single object, centred, well lit, on a plain
white background, photographed at a consistent distance. Real BinSight input
will differ in ways that reliably cost accuracy:

* **Clutter** — several items in frame, or an item on a patterned worktop. The
  model has never seen a background that carries information.
* **Lighting** — indoor tungsten, evening light, on-device flash, and shadows
  cast by the phone itself. Colour temperature shifts hit glass-vs-plastic
  hardest, since both are largely judged on transparency and specular highlights.
* **Partial objects** — a bottle held close enough to crop the neck, or an item
  half inside a bin.
* **Packaging locality** — TrashNet is one collector's American/European
  packaging from the mid-2010s. Composite materials (coffee cups, crisp packets,
  Tetra Pak) barely exist in it, and those are exactly the items people are
  unsure about.
* **Class definition** — `general_waste` is TrashNet's `trash`, the smallest
  class at 137 images. It is a residual category, not a material, so it has the
  least coherent visual definition and the least data.

Expect a substantial drop from the test number on real handheld photos. Nothing
in this notebook measures that drop; measuring it needs a small hand-collected
BinSight evaluation set, which does not exist yet.

**This model is a portfolio demonstration. It is not production-ready, and the
app must not present its output as authoritative disposal advice.**

## 9. Labels — derived from the pipeline, then verified

The output order is not assumed to be alphabetical. It is fixed by
`CLASS_ORDER`, which is what `label_index` was built from in section 3, and the
assertions below re-derive it from the manifest and from the model itself.

In [ ]:
# Re-derive the index -> label mapping from the manifest rather than trusting
# the constant, then check the two agree.
derived = (
    manifest[["label_index", "label"]]
    .drop_duplicates()
    .sort_values("label_index")["label"]
    .tolist()
)

assert derived == CLASS_ORDER, f"manifest order {derived} != CLASS_ORDER {CLASS_ORDER}"
assert model.output_shape[-1] == len(CLASS_ORDER), "model output width != number of labels"
assert derived != sorted(derived), (
    "The derived order is alphabetical, which means class_names were probably "
    "sorted somewhere instead of taken from CLASS_ORDER."
)
print("Label order verified against the manifest and the model output width.\n")

LABELS_PATH = ARTIFACTS / "labels.txt"
LABELS_PATH.write_text("\n".join(CLASS_ORDER) + "\n", encoding="utf-8")
print(f"Saved {LABELS_PATH}:")
for i, name in enumerate(CLASS_ORDER):
    print(f"  index {i}: {name}")

## 10. TFLite / LiteRT export

Keras 3 models convert most reliably via an exported TF SavedModel, so the
notebook takes that route and falls back to `from_keras_model` only if the
export path fails.

`SELECT_TF_OPS` is deliberately **not** enabled. The converter is pinned to
builtin ops only; if a model needs Flex delegates it cannot run on the simple
iOS LiteRT runtime, and that is a blocker to surface rather than paper over.

In [ ]:
SAVED_MODEL_DIR = WORK / "saved_model"
if SAVED_MODEL_DIR.exists():
    shutil.rmtree(SAVED_MODEL_DIR)

model.export(str(SAVED_MODEL_DIR))
print(f"Exported SavedModel to {SAVED_MODEL_DIR}")


def make_converter():
    try:
        converter = tf.lite.TFLiteConverter.from_saved_model(str(SAVED_MODEL_DIR))
    except Exception as exc:  # noqa: BLE001
        print(f"from_saved_model failed ({type(exc).__name__}: {exc}); using from_keras_model")
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
    # Builtin ops only - no Flex/SELECT_TF_OPS.
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
    return converter


def representative_dataset():
    """Real training images, used to calibrate int8 activation ranges."""
    taken = 0
    for images, _ in train_ds.unbatch().batch(1):
        yield [tf.cast(images, tf.float32)]
        taken += 1
        if taken >= 200:
            break


CANDIDATES = {}

In [ ]:
# --- float32 baseline --------------------------------------------------------
try:
    converter = make_converter()
    CANDIDATES["float32"] = converter.convert()
    print(f"float32 : {len(CANDIDATES['float32']) / 1e6:.2f} MB")
except Exception as exc:  # noqa: BLE001
    print(f"float32 conversion FAILED: {type(exc).__name__}: {exc}")
    print("If this mentions SELECT_TF_OPS or Flex ops, treat it as a BLOCKER and")
    print("report it rather than enabling SELECT_TF_OPS - the iOS runtime cannot use it.")
    raise

In [ ]:
# --- float16 post-training quantisation --------------------------------------
# Preferred mobile candidate: half the size, and the input/output tensors stay
# float32, so the iOS contract is unchanged.
try:
    converter = make_converter()
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    CANDIDATES["float16"] = converter.convert()
    print(f"float16 : {len(CANDIDATES['float16']) / 1e6:.2f} MB")
except Exception as exc:  # noqa: BLE001
    print(f"float16 conversion failed: {type(exc).__name__}: {exc}")

In [ ]:
# --- full int8 (optional) ----------------------------------------------------
# Kept float32 at the boundaries on purpose: fully-quantised int8 I/O would push
# scale/zero-point handling into the Swift code for very little extra saving.
if CONFIG["export_int8"]:
    try:
        converter = make_converter()
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        CANDIDATES["int8"] = converter.convert()
        print(f"int8    : {len(CANDIDATES['int8']) / 1e6:.2f} MB")
    except Exception as exc:  # noqa: BLE001
        print(f"int8 conversion failed (not fatal, it is an optional candidate): "
              f"{type(exc).__name__}: {exc}")

for name, blob in CANDIDATES.items():
    (WORK / f"candidate_{name}.tflite").write_bytes(blob)
print(f"\nCandidates produced: {list(CANDIDATES)}")

### 10.1 Validate each candidate with the LiteRT interpreter

Every candidate is run on real held-out test images, then compared numerically
against the Keras model on the same batch. Conversion "succeeding" is not
evidence the model still works — a silently wrong quantisation produces a valid
file that predicts nonsense.

In [ ]:
COMPARISON_N = min(128, len(test_frame))
comparison_images = np.concatenate([x.numpy() for x, _ in test_ds])[:COMPARISON_N]
comparison_truth = test_truth[:COMPARISON_N]
keras_probabilities = model.predict(comparison_images, verbose=0)
keras_top1 = keras_probabilities.argmax(axis=1)


def run_interpreter(blob, images):
    interpreter = tf.lite.Interpreter(model_content=blob)
    interpreter.allocate_tensors()
    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    outputs = []
    for image in images:
        sample = np.expand_dims(image, 0)
        if input_detail["dtype"] == np.int8:
            scale, zero_point = input_detail["quantization"]
            sample = np.round(sample / scale + zero_point).astype(np.int8)
        elif input_detail["dtype"] == np.uint8:
            scale, zero_point = input_detail["quantization"]
            sample = np.round(sample / scale + zero_point).astype(np.uint8)
        else:
            sample = sample.astype(input_detail["dtype"])

        interpreter.set_tensor(input_detail["index"], sample)
        interpreter.invoke()
        result = interpreter.get_tensor(output_detail["index"])[0].astype(np.float32)
        if output_detail["dtype"] in (np.int8, np.uint8):
            scale, zero_point = output_detail["quantization"]
            result = (result - zero_point) * scale
        outputs.append(result)

    return np.stack(outputs), input_detail, output_detail


VALIDATION = {}
for name, blob in CANDIDATES.items():
    probabilities, input_detail, output_detail = run_interpreter(blob, comparison_images)
    top1 = probabilities.argmax(axis=1)

    VALIDATION[name] = {
        "size_bytes": len(blob),
        "size_mb": round(len(blob) / 1e6, 3),
        "input_dtype": np.dtype(input_detail["dtype"]).name,
        "input_shape": [int(v) for v in input_detail["shape"]],
        "input_quantization": {
            "scale": float(input_detail["quantization"][0]),
            "zero_point": int(input_detail["quantization"][1]),
        },
        "output_dtype": np.dtype(output_detail["dtype"]).name,
        "output_shape": [int(v) for v in output_detail["shape"]],
        "output_quantization": {
            "scale": float(output_detail["quantization"][0]),
            "zero_point": int(output_detail["quantization"][1]),
        },
        "max_abs_prob_diff_vs_keras": float(np.max(np.abs(probabilities - keras_probabilities))),
        "mean_abs_prob_diff_vs_keras": float(np.mean(np.abs(probabilities - keras_probabilities))),
        "top1_agreement_vs_keras": float(np.mean(top1 == keras_top1)),
        "accuracy_on_comparison_batch": float(accuracy_score(comparison_truth, top1)),
    }

    print(f"\n--- {name} ---")
    for key, value in VALIDATION[name].items():
        print(f"  {key}: {value}")

print(f"\nKeras accuracy on the same {COMPARISON_N}-image batch: "
      f"{accuracy_score(comparison_truth, keras_top1):.4f}")

### 10.2 Select the final candidate

Not on accuracy alone. The selection rule, applied in order:

1. Reject any candidate whose top-1 agreement with Keras is below 99%.
2. Among survivors, prefer a **float** input/output contract — an int8 boundary
   pushes scale and zero-point arithmetic into Swift for a saving that does not
   matter at this model size.
3. Among those, take the smallest file.

float16 is expected to win: roughly half the size of float32, numerically almost
identical, and no change to the iOS contract.

In [ ]:
AGREEMENT_FLOOR = 0.99
FLOAT_DTYPES = {"float32", "float16"}

eligible = []
for name, stats in VALIDATION.items():
    reasons = []
    if stats["top1_agreement_vs_keras"] < AGREEMENT_FLOOR:
        reasons.append(
            f"top-1 agreement {stats['top1_agreement_vs_keras']:.4f} < {AGREEMENT_FLOOR}"
        )
    simple_io = stats["input_dtype"] in FLOAT_DTYPES and stats["output_dtype"] in FLOAT_DTYPES
    eligible.append({
        "name": name,
        "size_bytes": stats["size_bytes"],
        "simple_float_io": simple_io,
        "agreement": stats["top1_agreement_vs_keras"],
        "rejected_because": reasons,
    })

survivors = [c for c in eligible if not c["rejected_because"]]
assert survivors, f"No candidate met the agreement floor: {eligible}"

survivors.sort(key=lambda c: (not c["simple_float_io"], c["size_bytes"]))
SELECTED = survivors[0]["name"]

print("Candidate ranking:\n")
for candidate in eligible:
    status = "REJECTED " + "; ".join(candidate["rejected_because"]) if candidate["rejected_because"] else "eligible"
    marker = " <-- SELECTED" if candidate["name"] == SELECTED else ""
    print(
        f"  {candidate['name']:8s} {candidate['size_bytes'] / 1e6:6.2f} MB  "
        f"float-io={candidate['simple_float_io']!s:5s} agreement={candidate['agreement']:.4f}  "
        f"{status}{marker}"
    )

MODEL_PATH = ARTIFACTS / "BinSightWasteClassifier.tflite"
MODEL_PATH.write_bytes(CANDIDATES[SELECTED])
print(f"\nSelected {SELECTED!r} -> {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e6:.2f} MB)")

## 11. Model metadata

The documented tool for embedding TFLite Model Metadata is `tflite-support`.
Its latest release (0.4.4) publishes wheels only up to **cp311**, and the Colab
runtime is **Python 3.12** — so on Colab this install is expected to fail.

The notebook attempts it anyway (it succeeds on a 3.11 runtime) and otherwise
falls back to the sidecar `labels.txt` + `model_contract.json`, which is what
the iOS app reads regardless. The outcome is recorded honestly in the contract,
so nobody has to guess whether metadata is embedded.

In [ ]:
METADATA = {"embedded": False, "tool": None, "reason": None}

try:
    import tflite_support  # noqa: F401
    from tflite_support.metadata_writers import image_classifier as mw_image_classifier
    from tflite_support.metadata_writers import writer_utils

    label_file = WORK / "labels_for_metadata.txt"
    label_file.write_text("\n".join(CLASS_ORDER) + "\n", encoding="utf-8")

    writer = mw_image_classifier.MetadataWriter.create_for_inference(
        writer_utils.load_file(str(MODEL_PATH)),
        input_norm_mean=[0.0, 0.0, 0.0],   # normalisation is baked into the graph
        input_norm_std=[1.0, 1.0, 1.0],
        label_file_paths=[str(label_file)],
    )
    writer_utils.save_file(writer.populate(), str(MODEL_PATH))
    METADATA = {
        "embedded": True,
        "tool": f"tflite-support {tflite_support.__version__}",
        "reason": None,
    }
    print(f"Embedded TFLite Model Metadata using tflite-support {tflite_support.__version__}")
except Exception as exc:  # noqa: BLE001
    METADATA = {
        "embedded": False,
        "tool": None,
        "reason": f"{type(exc).__name__}: {exc}",
    }
    print("TFLite Model Metadata NOT embedded.")
    print(f"  reason: {METADATA['reason']}")
    print("  Expected on Python 3.12: tflite-support 0.4.4 ships no cp312 wheel.")
    print("  Falling back to labels.txt + model_contract.json, which the app reads anyway.")

## 12. Contract, checksums and metrics

In [ ]:
selected_stats = VALIDATION[SELECTED]
sha256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()

MODEL_CONTRACT = {
    "schema_version": 1,
    "verified_by_execution": True,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model": {
        "name": "BinSightWasteClassifier",
        "version": "1.0.0",
        "file": MODEL_PATH.name,
        "description": "Six-class waste material image classifier for the BinSight iOS app.",
        "author": "Maryna Antonevych",
        "quantization": SELECTED,
        "size_bytes": selected_stats["size_bytes"],
        "sha256": sha256,
        "metadata_embedded": METADATA["embedded"],
        "metadata_tool": METADATA["tool"],
        "metadata_skip_reason": METADATA["reason"],
    },
    "input": {
        "name": "image",
        "shape": selected_stats["input_shape"],
        "dtype": selected_stats["input_dtype"],
        "color_space": "RGB",
        "value_range": [0.0, 255.0],
        "layout": "NHWC",
        "resize": "bilinear to 224x224, aspect ratio not preserved",
        "preprocessing_note": (
            "Normalisation is baked into the graph. Pass raw 0-255 RGB float32 "
            "pixels; do NOT apply ImageNet mean/std on the client."
        ),
        "baked_normalization": NORMALIZATION,
        "quantization": selected_stats["input_quantization"],
    },
    "output": {
        "name": "probabilities",
        "shape": selected_stats["output_shape"],
        "dtype": selected_stats["output_dtype"],
        "activation": "softmax",
        "quantization": selected_stats["output_quantization"],
        "labels_in_index_order": CLASS_ORDER,
        "display_names": INTERNAL_TO_DISPLAY,
    },
    "numerical_validation": {
        "comparison_batch_size": COMPARISON_N,
        "max_abs_prob_diff_vs_keras": selected_stats["max_abs_prob_diff_vs_keras"],
        "mean_abs_prob_diff_vs_keras": selected_stats["mean_abs_prob_diff_vs_keras"],
        "top1_agreement_vs_keras": selected_stats["top1_agreement_vs_keras"],
    },
    "training": {
        "backbone_preset": PRESET,
        "selected_stage": SELECTED_STAGE,
        "seed": SEED,
        "dataset": "TrashNet (dataset-resized.zip)",
        "dataset_url": DATASET_URL,
        "dataset_licence": "MIT (garythung/trashnet); citation of the repository requested",
    },
    "runtime": {
        "requires_select_tf_ops": False,
        "target_ops": "TFLITE_BUILTINS",
    },
}

CONTRACT_PATH = ARTIFACTS / "model_contract.json"
CONTRACT_PATH.write_text(json.dumps(MODEL_CONTRACT, indent=2) + "\n", encoding="utf-8")
print(json.dumps(MODEL_CONTRACT, indent=2))
print(f"\nSaved {CONTRACT_PATH}")

In [ ]:
METRICS = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "environment": ENVIRONMENT,
    "config": CONFIG,
    "dataset": {
        "name": "TrashNet",
        "url": DATASET_URL,
        "licence": "MIT (garythung/trashnet)",
        "total_images_discovered": int(len(frame)),
        "unreadable_files": len(corrupt),
        "class_counts": {row["label"]: int(row["count"]) for _, row in distribution.iterrows()},
        "imbalance_ratio": float(IMBALANCE_RATIO),
        "split_sizes": {
            "train": int(len(train_frame)),
            "val": int(len(val_frame)),
            "test": int(len(test_frame)),
        },
    },
    "training": {
        "selected_stage": SELECTED_STAGE,
        "class_weighting_applied": CLASS_WEIGHT is not None,
        "stage_a_best_val_accuracy": stage_a_val_accuracy,
        "stage_a_minutes": round(stage_a_seconds / 60, 2),
        "stage_a_epochs_run": len(history_a.history["loss"]),
        "stage_b_enabled": bool(CONFIG["stage_b_enabled"]),
        "stage_b_best_val_accuracy": stage_b_val_accuracy,
        "stage_b_minutes": round(stage_b_seconds / 60, 2),
        "stage_b_kept": used_stage_b,
    },
    "test": {
        "n": int(len(test_truth)),
        "accuracy": test_accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "per_class": {
            name: {
                "precision": report_dict[name]["precision"],
                "recall": report_dict[name]["recall"],
                "f1": report_dict[name]["f1-score"],
                "support": int(report_dict[name]["support"]),
            }
            for name in CLASS_ORDER
        },
        "confusion_matrix_counts": confusion_counts.tolist(),
        "labels_in_index_order": CLASS_ORDER,
    },
    "export": {
        "selected_candidate": SELECTED,
        "candidates": VALIDATION,
        "metadata": METADATA,
    },
}

METRICS_PATH = ARTIFACTS / "metrics.json"
METRICS_PATH.write_text(json.dumps(METRICS, indent=2) + "\n", encoding="utf-8")
print(f"Saved {METRICS_PATH}")
print(json.dumps({"test": METRICS["test"]["accuracy"],
                  "macro_f1": METRICS["test"]["macro_f1"],
                  "selected": SELECTED}, indent=2))

In [ ]:
# --- Checksums ---------------------------------------------------------------
CHECKSUM_FILES = [
    MODEL_PATH,
    ARTIFACTS / "labels.txt",
    CONTRACT_PATH,
    METRICS_PATH,
    ARTIFACTS / "classification_report.csv",
    ARTIFACTS / "split_manifest.csv",
    ARTIFACTS / "confusion_matrix.png",
    ARTIFACTS / "confusion_matrix_normalized.png",
    ARTIFACTS / "training_curves.png",
]

lines = []
for path in CHECKSUM_FILES:
    if not path.exists():
        print(f"  (missing, skipped): {path.name}")
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    lines.append(f"{digest}  {path.name}")

CHECKSUMS_PATH = ARTIFACTS / "checksums.sha256"
CHECKSUMS_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Saved {CHECKSUMS_PATH}:\n")
print("\n".join(lines))

## 13. Collect the artefacts

Download `binsight_artifacts.zip` and unpack it over `ml/artifacts/` in the
repository. `ml/README.md` lists which files to commit and which to leave out.

In [ ]:
bundle = shutil.make_archive(str(WORK / "binsight_artifacts"), "zip", root_dir=str(ARTIFACTS))
print(f"Bundle: {bundle} ({pathlib.Path(bundle).stat().st_size / 1e6:.2f} MB)\n")
print("Contents:")
for path in sorted(ARTIFACTS.iterdir()):
    print(f"  {path.name:38s} {path.stat().st_size / 1024:9.1f} KB")

try:
    from google.colab import files  # type: ignore
    files.download(bundle)
except Exception:  # noqa: BLE001
    print(f"\nNot running in Colab - copy {bundle} manually.")

print(f"""
Next steps
----------
1. Unpack binsight_artifacts.zip into ml/artifacts/ in the repository.
2. Copy ml/artifacts/BinSightWasteClassifier.tflite and ml/artifacts/labels.txt
   into the iOS app bundle when Phase 4 wires up inference.
3. Fill docs/MODEL_CARD.md from ml/artifacts/metrics.json.
4. Record the run duration and the GPU type from section 1 in the Model Card.

Selected model : {SELECTED}
Test accuracy  : {test_accuracy:.4f}
Macro F1       : {macro_f1:.4f}
SHA256         : {sha256}
""")